Luis Anunciacao, PhD
2026-10-15 - 20:45

# DP - AGEM Time Series

Raw sources: the 10 SurveyMonkey exports in `Original data and metadata/`, read from this GitHub repository.
Output: `Importable data/df.rds`.

## §1 Records

In [1]:
pacman::p_load(readxl, tidyverse, janitor, stringi)

# column -> item maps for the columns after the 9 SurveyMonkey metadata columns; items follow T1 numbering
# (T2-only items: lideres 21-22, emocional 50-54). c(15, 16) = one T2 column matched to two T1 items (lideres T2 "contratar e desligar");
# NA = column not used (free text of "outro"). Lideres T3 positions follow the current SurveyMonkey labels (= T1 order).
lid_t1 <- as.list(1:20)
lid_t2 <- list(1, 2, 3, 5, 4, 6, 7, 8, 9, 11, 12, 14, 13, c(15, 16), 17, 21, 18, 19, 20, 22)
lid_t3 <- c(as.list(1:20), list(23, NA, 24), as.list(rep(25, 14)))
emo_t1 <- as.list(c(1:49, rep(55, 5), 56, 57))
emo_t2 <- as.list(c(1, 2, 3, 4, 6, 7, 9, 10, 15:35, 50, 36:39, 51, 40:43, 52, 44:47, 53, 48, 54, 49, 57, 58, rep(59, 10)))

df_original <- tribble(
  ~instrument, ~company, ~wave, ~file, ~col_map,
  "lideres", "finx", "T1", "T1 FINX - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.xlsx", lid_t1,
  "lideres", "finx", "T2", "T2 FINX - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.csv", lid_t2,
  "lideres", "finx", "T3", "T3 FINX - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.csv", lid_t3,
  "lideres", "xforce", "T1", "T1 Xforce - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.xlsx", lid_t1,
  "lideres", "xforce", "T2", "T2 Xforce - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.xlsx", lid_t2,
  "lideres", "xforce", "T3", "T3 XFORCE - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.xlsx", lid_t3,
  "emocional", "finx", "T1", "T1 FINX - INVENTÁRIO DE SAÚDE EMOCIONAL.xlsx", emo_t1,
  "emocional", "finx", "T2", "T2 FINX - INVENTÁRIO DE SAÚDE EMOCIONAL.xlsx", emo_t2,
  "emocional", "xforce", "T1", "T1 Xforce- INVENTÁRIO DE SAÚDE EMOCIONAL.xlsx", emo_t1,
  "emocional", "xforce", "T2", "T2 XFORCE - INVENTÁRIO DE SAÚDE EMOCIONAL.xlsx", emo_t2) |>
  # downloads each SurveyMonkey export from GitHub (Original data and metadata/) and reads it with no header and every cell as text;
  # private repository, token from the git credential store (gitcreds)
  mutate(raw = map(file, \(f) curl::curl_download(paste0("https://raw.githubusercontent.com/anovabr/AGEM-Time-Series/main/Original%20data%20and%20metadata/", URLencode(f, reserved = TRUE)),
                                                  tempfile(fileext = paste0(".", tools::file_ext(f))),
                                                  handle = curl::new_handle() |> (\(h) tryCatch(curl::handle_setheaders(h, Authorization = paste("token", gitcreds::gitcreds_get()$password)), error = \(e) h))()) |>
                     (\(path) if (tools::file_ext(f) == "csv") read_csv(path, col_names = FALSE, col_types = cols(.default = "c"))
                               else read_excel(path, col_names = FALSE, col_types = "text", .name_repair = "unique_quiet"))()))

rm(lid_t1, lid_t2, lid_t3, emo_t1, emo_t2)

df_original |> transmute(instrument, company, wave, rows = map_int(raw, nrow), cols = map_int(raw, ncol))

instrument,company,wave,rows,cols
<chr>,<chr>,<chr>,<int>,<int>
lideres,finx,T1,17,29
lideres,finx,T2,18,29
lideres,finx,T3,17,46
lideres,xforce,T1,7,29
lideres,xforce,T2,8,29
lideres,xforce,T3,5,46
emocional,finx,T1,88,65
emocional,finx,T2,88,69
emocional,xforce,T1,27,65


## §2 Analytic frame

One row per respondent × answered item, lowercased and de-accented.

- Row 1 of each export is the question text; exports with a `Response` sub-header have 2 header rows.
- `item_text` is the wording of the earliest wave that has the item (the T3 export headers are outdated, so T3 text is used only for T3-only items).

In [2]:
df <- df_original |>
  # each export to long format: respondent x answered column, with question text (carried right over multiple-choice columns),
  # item from the map and the column position inside its item (option order of multiple-choice items)
  mutate(long = map2(raw, col_map, \(raw, col_map) {
    stopifnot(ncol(raw) - 9 == length(col_map))   # the map must cover every item column
    raw |>
      slice(-seq_len(if (is.na(raw[[1]][2])) 2 else 1)) |>
      select(respondent_id = 1, 10:ncol(raw)) |>
      pivot_longer(-respondent_id, names_to = "col", values_to = "response", values_drop_na = TRUE) |>
      mutate(col = match(col, names(raw)) - 9,
             item_text = (tibble(t = unlist(raw[1, -(1:9)])) |> fill(t) |> pull(t))[col],
             item = col_map[col],
             position = col - match(item, col_map) + 1) |>
      unnest(item) |>
      filter(!is.na(item)) |>
      select(-col)
  })) |>
  select(instrument, company, wave, long) |>
  unnest(long) |>
  clean_names() |>
  mutate(across(where(is.character), ~ str_squish(stri_trans_general(tolower(.), "Latin-ASCII")))) |>
  remove_empty("rows") |>
  mutate(wave = toupper(wave)) |>
  mutate(item_text = first(item_text[order(wave)], na_rm = TRUE), .by = c(instrument, item))

df |> count(instrument, company, wave, name = "answers")

instrument,company,wave,answers
<chr>,<chr>,<chr>,<int>
emocional,finx,T1,4605
emocional,finx,T2,4593
emocional,xforce,T1,1334
emocional,xforce,T2,1006
lideres,finx,T1,275
lideres,finx,T2,314
lideres,finx,T3,462
lideres,xforce,T1,119
lideres,xforce,T2,125


## §3 Recodes

- `code`: scale position (1-5) for scale items (lideres 1-22, emocional 1-54); option order for categorical items; column order for multiple-choice options.
- `test_value`: score used in the tests (scale items only; = `code`).
- `risk`: risk items, i.e. negatively worded scale items (higher agreement or frequency = higher risk): lideres 1, 4, 5, 7, 8, 12, 18, 19, 21, 22; emocional 4-30, 39, 50-53.
- `code_3` / `response_3`: scale responses collapsed to 3 categories (1-2 -> 1, 3 -> 2, 4-5 -> 3), with English labels (disagree / neutral / agree; never or rarely / sometimes / often or always); tenure keeps its 6 categories, labelled in English.
- `item_en` / `label`: English translation of the item wording (first wave) and a short English label for figures.
- Stable items only: items worded identically in every wave on the SurveyMonkey forms used for collection. Lideres (T1-T3): 1, 3, 7, 8, 9, 12, 14, 19. Emocional (T1-T2): 1-4, 6, 7, 10, 15-38, 40, 42-47, plus tenure (57) as a covariate.
  Excluded lideres items: 2, 4, 5, 6, 11, 13, 15-18, 20 (reworded or negated in T2), 10 (not asked in T2), 21-22 (T2 only), 23-25 (T3 only).
  Excluded emocional items: 9, 39, 41, 48, 49 (reworded in T2), 5, 8, 11-14, 55, 56 (T1 only), 50-54, 58, 59 (T2 only).

In [3]:
likert <- list(lideres = c("discordo totalmente", "discordo parcialmente", "nem concordo nem discordo", "concordo parcialmente", "concordo totalmente"),
               emocional = c("nunca", "raramente", "as vezes", "muitas vezes", "sempre"))
likert_max <- c(lideres = 22, emocional = 54)
categories <- list("lideres 23" = c("administrativo", "comercial/vendas", "financeiro", "recursos humanos", "operacional/producao", "marketing",
                                    "tecnologia/informatica", "outro (especifique)"),
                   "lideres 24" = c("menos de 3 meses", "de 3 a 6 meses", "de 6 meses a 1 ano", "de 1 a 3 anos", "de 3 a 5 anos", "mais de 5 anos"),
                   "emocional 56" = c("sim", "nao"),
                   "emocional 57" = c("ate 6 meses", "de 6 meses a 1 ano", "de 1 ano a 2 anos", "de 2 anos a 4 anos", "4 anos a 6 anos", "mais de 6 anos"),
                   "emocional 58" = c("administrativo / financeiro", "apoio / servicos gerais", "comercial / atendimento", "operacional / producao"))

# adds code_3 (scale responses collapsed to 3 categories: 1-2, 3, 4-5) and response_3 (English label of code_3; tenure keeps its 6 categories)
label_responses <- \(x) x |>
  mutate(code_3 = if_else(item <= likert_max[instrument], c(1L, 1L, 2L, 3L, 3L)[code], code),
         response_3 = case_match(response,
                                 c("discordo totalmente", "discordo parcialmente") ~ "disagree", "nem concordo nem discordo" ~ "neutral",
                                 c("concordo parcialmente", "concordo totalmente") ~ "agree",
                                 c("nunca", "raramente") ~ "never or rarely", "as vezes" ~ "sometimes", c("muitas vezes", "sempre") ~ "often or always",
                                 "ate 6 meses" ~ "up to 6 months", "de 6 meses a 1 ano" ~ "6 months to 1 year", "de 1 ano a 2 anos" ~ "1 to 2 years",
                                 "de 2 anos a 4 anos" ~ "2 to 4 years", "4 anos a 6 anos" ~ "4 to 6 years", "mais de 6 anos" ~ "more than 6 years"))

df <- df |>
  mutate(code = case_when(item <= likert_max[instrument] ~ map2_int(response, instrument, \(r, i) match(r, likert[[i]])),
                          paste(instrument, item) %in% names(categories) ~ map2_int(response, paste(instrument, item), \(r, k) match(r, categories[[k]])),
                          .default = as.integer(position)),
         test_value = if_else(item <= likert_max[instrument], code, NA),
         risk = if_else(instrument == "lideres", item %in% c(1, 4, 5, 7, 8, 12, 18, 19, 21, 22), item %in% c(4:30, 39, 50:53))) |>
  select(-position) |>
  # stable items only (identical wording in every wave); emocional 57 = tenure, kept as covariate
  filter(if_else(instrument == "lideres", item %in% c(1, 3, 7, 8, 9, 12, 14, 19), item %in% c(1:4, 6, 7, 10, 15:38, 40, 42:47, 57))) |>
  label_responses() |>
  # English wording (translation of the first wave) and short labels for figures
  left_join(tribble(
    ~instrument, ~item, ~label, ~item_en,
    "lideres", 1, "High decision responsibility", "I feel that the burden of responsibility for decisions is high.",
    "lideres", 3, "Clear responsibilities and authority", "My responsibilities and limits of authority are clear (e.g., I know how far I can decide alone and when I need to escalate).",
    "lideres", 7, "Frequent conflict intervention", "I often need to intervene in conflicts that wear down my work routine.",
    "lideres", 8, "Team overly dependent on me", "I feel that my team depends excessively on me to solve simple problems (e.g., recurring questions that could be solved without direct supervision).",
    "lideres", 9, "Satisfied with team performance", "I am satisfied with my team's overall performance.",
    "lideres", 12, "Team unpreparedness raises my stress", "The team's lack of technical or behavioral preparation increases my stress level.",
    "lideres", 14, "Support from senior management", "I receive feedback and support from senior management when needed.",
    "lideres", 19, "Decisions affecting others", "My work requires constant decisions that affect the safety, well being or performance of other people (e.g., reallocating people, giving difficult feedback, setting priorities).",
    "emocional", 1, "Copes with frustration", "Do you cope well with frustrations and unexpected events? (e.g., staying calm even when you miss a deadline)",
    "emocional", 2, "Self control under stress", "Do you keep your self control in situations of stress or conflict? (e.g., not losing your temper in arguments)",
    "emocional", 3, "Assertive without aggression", "Can you be assertive without raising your voice or being aggressive? (e.g., saying what you need to say without shouting)",
    "emocional", 4, "Excessive self demand", "Do you demand too much of yourself? (e.g., feeling that you have never done enough)",
    "emocional", 6, "Anticipatory worry", "Do you suffer in anticipation? (e.g., worrying about something that has not happened yet)",
    "emocional", 7, "Rumination", "Do you dwell on past situations? (e.g., often recalling old hurts)",
    "emocional", 10, "Shyness or insecurity", "Do you consider yourself shy or insecure?",
    "emocional", 15, "Headache or muscle pain", "Do you have headaches or muscle pain?",
    "emocional", 16, "Tiredness", "Do you often feel tired? (e.g., waking up without energy even after sleeping)",
    "emocional", 17, "Sleep difficulty", "Do you have difficulty sleeping?",
    "emocional", 18, "Low motivation", "Do you feel unmotivated or discouraged? (e.g., no desire to start simple tasks)",
    "emocional", 19, "Sad or depressed mood", "Do you perceive yourself as sad or in a depressed mood? (e.g., lack of pleasure in daily activities)",
    "emocional", 20, "Difficulty concentrating", "Do you often have difficulty concentrating? (e.g., starting tasks and not finishing them)",
    "emocional", 21, "Forgetfulness", "Have you been forgetful or had memory lapses? (e.g., forgetting appointments or names)",
    "emocional", 22, "Stomach or bowel problems", "Do you have stomach aches or bowel problems?",
    "emocional", 23, "Feelings of inferiority", "Do you have feelings of inferiority? (e.g., believing you are not good enough)",
    "emocional", 24, "Decreased libido", "Do you notice decreased libido? (e.g., loss of sexual interest)",
    "emocional", 25, "Appetite change", "Do you have a lack or excess of appetite? (e.g., raiding the fridge at night or skipping meals)",
    "emocional", 26, "Anxiety or distress", "Do you often feel anxiety or distress? (e.g., tightness in the chest for no clear reason)",
    "emocional", 27, "Hair loss", "Do you have hair loss with no apparent cause?",
    "emocional", 28, "Disturbing thoughts", "Do you have disturbing thoughts? (e.g., imagining negative scenarios or distressing thoughts)",
    "emocional", 29, "Crying more", "Have you been crying more lately? (e.g., crying over small things)",
    "emocional", 30, "Wish to stop living", "Do you feel like giving up on living? (e.g., thoughts about death or not going on)",
    "emocional", 31, "Satisfied with oneself", "Are you satisfied with yourself and your choices? (e.g., believing you are on the right path)",
    "emocional", 32, "Satisfied with role", "Are you satisfied with your current role? (e.g., feeling that your role suits you)",
    "emocional", 33, "Trusted to decide", "Does the company trust your ability to make decisions? (e.g., solving problems without asking permission for everything)",
    "emocional", 34, "Freedom to organize activities", "Are you free to organize your activities? (e.g., choosing the order of tasks)",
    "emocional", 35, "Encouraged to use creativity", "Are you encouraged to use your creativity and knowledge? (e.g., suggesting improvements or new ideas)",
    "emocional", 36, "Mutual respect", "Do employees respect and care about one another? (e.g., absence of humiliation or toxic competition)",
    "emocional", 37, "Sense of belonging", "Do you feel you belong to the team and the company? (e.g., feeling part of the group)",
    "emocional", 38, "Teamwork", "Is there a sense of teamwork? (e.g., collaboration among professionals)",
    "emocional", 40, "Equal opportunities", "Do employees have the same opportunities? (e.g., promotions without favoritism)",
    "emocional", 42, "Fair treatment", "Is everyone treated well regardless of personal differences? (e.g., absence of prejudice based on age, gender or ethnicity)",
    "emocional", 43, "Adequate equipment", "Do you have access to adequate equipment and materials to do your tasks?",
    "emocional", 44, "Comfortable talking to leader", "Do you feel comfortable talking to your leader? (e.g., expressing expectations and conflicts without fear)",
    "emocional", 45, "Receives feedback", "Do you receive feedback to improve your performance? (e.g., clear and useful guidance)",
    "emocional", 46, "Adequate training", "Do you receive adequate training? (e.g., learning what is needed to do your job)",
    "emocional", 47, "Leader shows concern", "Does your leader show concern for your well being? (e.g., asking how you are and offering support)",
    "emocional", 57, "Tenure", "How long have you been working at this company?"), by = c("instrument", "item"))

stopifnot(!anyNA(df$code), !anyNA(df$response_3), !anyNA(df$item_en))   # every answer matched a known option and every item has a translation

df |> count(instrument, code_3, response_3)

instrument,code_3,response_3,n
<chr>,<int>,<chr>,<int>
emocional,1,never or rarely,2138
emocional,1,up to 6 months,89
emocional,2,6 months to 1 year,48
emocional,2,sometimes,1886
emocional,3,1 to 2 years,33
emocional,3,often or always,4094
emocional,4,2 to 4 years,24
emocional,5,4 to 6 years,11
emocional,6,more than 6 years,5


## §4 Option list

Full option list per item, stored as `attr(df, "options")`: scale and categorical options including options nobody chose; multiple-choice options as observed.

In [4]:
attr(df, "options") <- bind_rows(distinct(df, instrument, item, code, response),
                                 distinct(df, instrument, item) |>
                                   mutate(response = map2(instrument, item, \(i, it) if (it <= likert_max[i]) likert[[i]] else categories[[paste(i, it)]])) |>
                                   unnest(response) |>
                                   mutate(code = row_number(), .by = c(instrument, item))) |>
  distinct() |>
  label_responses() |>
  arrange(instrument, item, code)

rm(likert, likert_max, categories, label_responses)

## §5 Save

In [5]:
saveRDS(df, "../Importable data/df.rds")